In [0]:
# %sql
# SELECT current_metastore();

In [0]:
%sql
-- DESCRIBE METASTORE;

In [0]:
#importing packages
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# dbutils.widgets.text("file_name","")

In [0]:
# p_file_name = dbutils.widgets.get("file_name")
# print(p_file_name)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medillian_upsert.bronze

In [0]:
# CELL 3 — Ingestion loop with awaitTermination + audit columns
for p_file_name in ["customers", "orders", "product", "region"]:
    df = spark.readStream.format("cloudFiles") \
            .option("cloudFiles.format", "parquet") \
            .option("pathGlobFilter", "*.parquet") \
            .option("cloudFiles.schemaLocation", f"/Volumes/medillian_upsert/default/{p_file_name}/{p_file_name}_schema") \
            .load(f"/Volumes/medillian_upsert/default/{p_file_name}/") \
            .withColumn("_ingested_at", current_timestamp()) \
            .withColumn("_source_file", col("_metadata.file_path"))

    query = df.writeStream.format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", f"/Volumes/medillian_upsert/default/{p_file_name}/{p_file_name}_checkpoint") \
            .trigger(availableNow=True) \
            .toTable(f"medillian_upsert.bronze.{p_file_name}")

    query.awaitTermination()  # wait for each stream to finish before next one

In [0]:
display(spark.read.table("medillian_upsert.bronze.customers"))

In [0]:
# for p_file_name in ["customers", "orders", "product", "region"]:
#     dbutils.fs.rm(f"/Volumes/medillian_upsert/default/{p_file_name}/{p_file_name}_schema", recurse=True)
#     dbutils.fs.rm(f"/Volumes/medillian_upsert/default/{p_file_name}/{p_file_name}_checkpoint", recurse=True)
#     dbutils.fs.rm(f"/Volumes/medillian_upsert/default/{p_file_name}/checkpoint", recurse=True)
#     dbutils.fs.rm(f"/Volumes/medillian_upsert/default/{p_file_name}", recurse=True)

In [0]:
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.bronze.customers")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.bronze.orders")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.bronze.product")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.bronze.region")

# spark.sql("DROP TABLE IF EXISTS medillian_upsert.silver.customers")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.silver.orders")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.silver.products")

# spark.sql("DROP TABLE IF EXISTS medillian_upsert.gold.dimcustomers")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.gold.dimproducts")
# spark.sql("DROP TABLE IF EXISTS medillian_upsert.gold.factorders")